# 面试问题：长时 Agent 怎样用 Event Sourcing 实现崩溃恢复？

可以直接复述的回答是：第一，把用户请求、计划、工具调用意图、工具结果和最终提交都记录为不可变事件。第二，当前状态由 reducer 重放得到，不依赖某个 worker 的内存。第三，工具调用必须使用稳定幂等键，恢复时先查调用结果再决定是否重试。第四，重复事件不能重复产生副作用。第五，只有持久化调用意图后才能调用外部系统。第六，通过崩溃点重放、重复投递和最终状态对照验证恢复能力。下面用差旅预订 Agent 演示。

## 真实案例：差旅 Agent 在酒店预订后崩溃

一名员工请求预订上海到北京的高铁和两晚酒店。事件字段参考持久化 Agent 运行时：run_id、event_id、type、call_id、payload 和 sequence。案例在酒店调用意图持久化后模拟进程崩溃；外部工具结果由离线字典代替。没有真实姓名、票号或支付信息。

In [1]:
request = {"run_id": "TRIP-301", "user_request": "预订 8 月 12 日上海到北京高铁，并订两晚公司协议酒店", "budget": 1800, "city": "北京", "nights": 2}  # 定义具有真实差旅字段的用户请求
seed_events = [  # 构造崩溃前已经持久化的五条事件
    {"seq": 1, "event_id": "e1", "type": "user_requested", "payload": request},  # 保存原始用户意图和预算
    {"seq": 2, "event_id": "e2", "type": "plan_created", "payload": {"steps": ["search_train", "book_train", "search_hotel", "book_hotel", "answer"]}},  # 保存可审计执行计划
    {"seq": 3, "event_id": "e3", "type": "tool_requested", "call_id": "train-search-1", "payload": {"tool": "train_search", "date": "2026-08-12"}},  # 先持久化高铁查询意图
    {"seq": 4, "event_id": "e4", "type": "tool_succeeded", "call_id": "train-search-1", "payload": {"train": "G12", "price": 553}},  # 保存高铁查询的确定性结果
    {"seq": 5, "event_id": "e5", "type": "tool_requested", "call_id": "hotel-book-1", "payload": {"tool": "hotel_book", "city": "北京", "nights": 2}},  # 酒店调用意图落盘后进程崩溃
]  # 结束崩溃点事件流
print("崩溃前事件：seq | type | call_id | payload")  # 输入预览直接展示持久化账本
for event in seed_events:  # 逐条输出五个已保存事件
    print(f"{event['seq']} | {event['type']:15} | {event.get('call_id', '-'):15} | {event['payload']}")  # 呈现恢复器真正读取的字段


崩溃前事件：seq | type | call_id | payload
1 | user_requested  | -               | {'run_id': 'TRIP-301', 'user_request': '预订 8 月 12 日上海到北京高铁，并订两晚公司协议酒店', 'budget': 1800, 'city': '北京', 'nights': 2}
2 | plan_created    | -               | {'steps': ['search_train', 'book_train', 'search_hotel', 'book_hotel', 'answer']}
3 | tool_requested  | train-search-1  | {'tool': 'train_search', 'date': '2026-08-12'}
4 | tool_succeeded  | train-search-1  | {'train': 'G12', 'price': 553}
5 | tool_requested  | hotel-book-1    | {'tool': 'hotel_book', 'city': '北京', 'nights': 2}


## Baseline / 基线：只保存 worker 内存状态

如果 worker 在调用酒店工具后、写入结果前崩溃，新 worker 看不到旧内存，只能再次调用。下面用调用计数展示重复副作用。

In [2]:
external_calls = []  # 记录基线对酒店系统产生的外部调用
def volatile_worker(crash_after_call):  # 模拟仅依赖进程内存的差旅 worker
    local_state = {"hotel_confirmation": None}  # 每次启动都创建空的易失状态
    external_calls.append("hotel-book-1")  # 调用酒店系统并产生可能收费的副作用
    confirmation = "H-BJ-7788"  # 模拟酒店系统已经成功生成确认号
    if crash_after_call:  # 在结果写回内存前模拟进程崩溃
        return None  # 丢失本次成功调用的本地证据
    local_state["hotel_confirmation"] = confirmation  # 未崩溃时才保存确认号
    return local_state  # 返回当前 worker 的易失状态
first_attempt = volatile_worker(True)  # 第一次调用成功后立即崩溃
second_attempt = volatile_worker(False)  # 新 worker 因无状态而重复调用
print("易失状态基线：第一次结果=", first_attempt)  # 展示崩溃导致的状态丢失
print("易失状态基线：第二次结果=", second_attempt)  # 展示重启后重新获得确认号
print("酒店外部调用次数=", len(external_calls), external_calls)  # 明确暴露重复副作用


易失状态基线：第一次结果= None
易失状态基线：第二次结果= {'hotel_confirmation': 'H-BJ-7788'}
酒店外部调用次数= 2 ['hotel-book-1', 'hotel-book-1']


## 核心实现：事件 reducer 与幂等恢复

Reducer 只负责把不可变事件投影为状态；恢复器找到没有对应 succeeded 事件的 call_id，再用同一个幂等键查询外部系统。

In [3]:
def reduce_events(events):  # 从零实现确定性的 Agent 事件状态投影
    state = {"status": "new", "plan": [], "pending": {}, "results": {}, "answer": None}  # 初始化可重建运行状态
    for event in sorted(events, key=lambda item: item["seq"]):  # 严格按持久化序号重放事件
        if event["type"] == "user_requested":  # 用户请求事件启动一次运行
            state["status"] = "planning"  # 把运行状态推进到计划阶段
        elif event["type"] == "plan_created":  # 计划事件给出后续可执行步骤
            state["plan"] = list(event["payload"]["steps"])  # 复制计划避免修改历史事件
            state["status"] = "running"  # 把运行状态推进到执行阶段
        elif event["type"] == "tool_requested":  # 工具意图事件表示调用已获准
            state["pending"][event["call_id"]] = event["payload"]  # 记录尚未获得结果的稳定调用号
        elif event["type"] == "tool_succeeded":  # 工具成功事件提供权威结果
            state["results"][event["call_id"]] = event["payload"]  # 保存结果供后续回答使用
            state["pending"].pop(event["call_id"], None)  # 从待完成集合移除当前调用
        elif event["type"] == "answer_committed":  # 最终提交事件结束运行
            state["answer"] = event["payload"]["text"]  # 保存已提交给用户的最终答复
            state["status"] = "completed"  # 标记运行已经不可重复提交
    return state  # 返回由事件唯一决定的当前状态
crashed_state = reduce_events(seed_events)  # 从崩溃前账本重建运行状态
print("重放后的状态：", crashed_state)  # 展示 pending 调用和已完成结果
print("需要恢复的 call_id：", sorted(crashed_state["pending"]))  # 明确指出恢复器下一步处理对象


重放后的状态： {'status': 'running', 'plan': ['search_train', 'book_train', 'search_hotel', 'book_hotel', 'answer'], 'pending': {'hotel-book-1': {'tool': 'hotel_book', 'city': '北京', 'nights': 2}}, 'results': {'train-search-1': {'train': 'G12', 'price': 553}}, 'answer': None}
需要恢复的 call_id： ['hotel-book-1']


## 失败案例与修正：重复结果事件不能重复推进状态

消息队列可能至少投递一次，同一 tool_succeeded 会重复出现。修正方法是在追加事件时按 event_id 去重，同时让 reducer 以 call_id 覆盖同一逻辑结果而非累计副作用。

In [4]:
external_result_store = {"hotel-book-1": {"hotel": "北京协议酒店", "confirmation": "H-BJ-7788", "price": 760}}  # 模拟支持幂等键查询的酒店权威系统
event_log = list(seed_events)  # 从崩溃前不可变事件开始恢复
known_event_ids = {event["event_id"] for event in event_log}  # 建立持久化事件去重索引
def append_once(event):  # 实现按 event_id 幂等追加的最小事件存储门禁
    if event["event_id"] in known_event_ids:  # 重复投递不能再次进入账本
        return False  # 返回未追加状态供监控计数
    event_log.append(event)  # 追加通过去重检查的新事件
    known_event_ids.add(event["event_id"])  # 更新事件身份索引
    return True  # 返回成功追加状态
hotel_result_event = {"seq": 6, "event_id": "e6", "type": "tool_succeeded", "call_id": "hotel-book-1", "payload": external_result_store["hotel-book-1"]}  # 使用原 call_id 查询已有酒店结果
first_append = append_once(hotel_result_event)  # 第一次恢复结果正常写入账本
duplicate_append = append_once(dict(hotel_result_event))  # 模拟消息队列重复投递相同事件
answer_event = {"seq": 7, "event_id": "e7", "type": "answer_committed", "payload": {"text": "已选 G12，高铁 553 元；酒店 H-BJ-7788，两晚 760 元；合计 1313 元。"}}  # 构造带确认号和总价的最终答复
append_once(answer_event)  # 在工具结果持久化后提交用户答案
recovered_state = reduce_events(event_log)  # 从完整事件账本重建最终状态
print(f"重复结果：首次追加={first_append}，再次追加={duplicate_append}")  # 展示至少一次投递下的幂等行为
print("恢复后 pending：", recovered_state["pending"])  # 展示所有工具调用均已闭合
print("恢复后答案：", recovered_state["answer"])  # 展示用户真正收到的可读结果


重复结果：首次追加=True，再次追加=False
恢复后 pending： {}
恢复后答案： 已选 G12，高铁 553 元；酒店 H-BJ-7788，两晚 760 元；合计 1313 元。


## 结果表：易失状态与事件恢复对照

In [5]:
volatile_tool_calls = len(external_calls)  # 记录易失状态方案产生的酒店调用次数
event_sourced_tool_calls = 1  # 权威系统只执行过一次 hotel-book-1 幂等调用
print("方案 | 酒店副作用次数 | 可恢复 pending | 最终状态")  # 输出同一崩溃场景下的方案对照
print(f"volatile | {volatile_tool_calls} | 否 | {second_attempt}")  # 展示内存方案的重复调用和局部结果
print(f"event_sourced | {event_sourced_tool_calls} | 是 | {recovered_state['status']}")  # 展示事件方案的幂等恢复结果
print("最终事件账本：")  # 引出完整且可审计的状态迁移记录
for event in event_log:  # 按序展示七条最终事件
    print(f"{event['seq']}:{event['type']}:{event.get('call_id', '-')}")  # 输出精简事件轨迹用于人工复核


方案 | 酒店副作用次数 | 可恢复 pending | 最终状态
volatile | 2 | 否 | {'hotel_confirmation': 'H-BJ-7788'}
event_sourced | 1 | 是 | completed
最终事件账本：
1:user_requested:-
2:plan_created:-
3:tool_requested:train-search-1
4:tool_succeeded:train-search-1
5:tool_requested:hotel-book-1
6:tool_succeeded:hotel-book-1
7:answer_committed:-


## 结果解读

崩溃前状态重放明确暴露 hotel-book-1 为 pending，而高铁查询结果仍然保留。恢复器用同一 call_id 查询到已有确认号，没有再次预订；重复 e6 被拒绝，最终账本只有一次酒店成功事件。事件溯源解决的是状态证据和副作用身份问题，不只是把 Python 字典写到磁盘。

## 生产边界

真实系统需要事务性 outbox、持久化唯一约束、事件 schema 版本、快照压缩、敏感字段加密、租约和灾备复制。外部工具必须真正支持幂等键或查询既有结果，否则恢复仍可能重复付费。教学例子没有模拟网络分区、多 worker 竞争和事件存储故障。

## 最小回归测试

In [6]:
assert len(seed_events) >= 5  # 保证案例包含足够完整的崩溃前事件流
assert "hotel-book-1" in crashed_state["pending"]  # 保证重放能识别未闭合酒店调用
assert volatile_tool_calls == 2  # 保证易失状态基线真实暴露重复副作用
assert duplicate_append is False  # 保证重复事件不会再次进入持久化账本
assert recovered_state["status"] == "completed"  # 保证恢复后的运行成功提交最终答案
assert len([event for event in event_log if event.get("call_id") == "hotel-book-1" and event["type"] == "tool_succeeded"]) == 1  # 保证酒店成功结果在账本中只出现一次
